# Retrieval, ablation and router evaluation

**Owner:** Aliyan (retrieval/ablation) and Sohaib (router)

Skeleton only: fill in the empty code cells; do not change contracts.

In [1]:
# Data paths: run this cell first.
from pathlib import Path

REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = REPO_DIR / "data" / "papers.parquet"
EMBEDDINGS_PATH = REPO_DIR / "data" / "embeddings.npy"
DATA_PATH, EMBEDDINGS_PATH


(PosixPath('/media/ibraheem/HDD/research_paper_discovery/data/papers.parquet'),
 PosixPath('/media/ibraheem/HDD/research_paper_discovery/data/embeddings.npy'))

## Load eval files

Read eval/eval_retrieval.json and eval/eval_routing.json.

In [2]:
import json

retrieval_eval = json.loads((REPO_DIR / "eval" / "eval_retrieval.json").read_text(encoding="utf-8"))
routing_eval = json.loads((REPO_DIR / "eval" / "eval_routing.json").read_text(encoding="utf-8"))
print({"retrieval_items": len(retrieval_eval), "routing_items": len(routing_eval)})

{'retrieval_items': 30, 'routing_items': 40}


## Score retrieval

Call hit_rate_at_k and mrr on the hand-labelled ids.

In [3]:
# Stage 4 setup (Ibraheem): make src importable from a repo-root nbconvert run.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_DIR / ".env")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("repo root on sys.path:", str(REPO_DIR))
print("LLM_MODEL:", os.getenv("LLM_MODEL"))


repo root on sys.path: /media/ibraheem/HDD/research_paper_discovery
LLM_MODEL: openai/gpt-oss-20b


In [4]:
from src.evaluate import hit_rate_at_k, mrr
from src.rag import retrieve

results = []
gold = []
for item in retrieval_eval:
    filters = item["filters"]
    docs = retrieve(
        item["question"], k=5,
        year_min=filters.get("year_min"), year_max=filters.get("year_max"),
        category=filters.get("category"), topic=filters.get("topic"),
    )
    results.append([doc["arxiv_id"] for doc in docs])
    gold.append(list(item.get("relevant_ids", [])))

print({
    "hit_rate_at_5": hit_rate_at_k(results, gold, 5) if any(gold) else "PRELIMINARY: no verified labels",
    "mrr": mrr(results, gold) if any(gold) else "PRELIMINARY: no verified labels",
})

Retrieval resources loaded: FAISS ntotal=119919, embeddings=(119919, 384), doc_topics rows=119919, papers=119919


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'hit_rate_at_5': 1.0, 'mrr': 0.9566666666666667}


## Run the ablation

Call src.evaluate.run_ablation on the eval questions.

In [5]:
from src.evaluate import run_ablation

ablation = run_ablation([item["question"] for item in retrieval_eval])
ablation

,question,variant,hit@5,rr,llm_calls
0,Find papers about low-resource machine transla...,with rewrite+grade,1.0,1.000000,4
1,Find papers about low-resource machine transla...,without rewrite+grade,1.0,1.000000,0
2,Show papers on 3D object detection from 2022 o...,with rewrite+grade,1.0,1.000000,4
3,Show papers on 3D object detection from 2022 o...,without rewrite+grade,1.0,1.000000,0
4,Which papers discuss graph learning in 2024?,with rewrite+grade,1.0,1.000000,4
5,Which papers discuss graph learning in 2024?,without rewrite+grade,1.0,0.250000,0
6,Find papers on diffusion models for image gene...,with rewrite+grade,1.0,1.000000,4
7,Find papers on diffusion models for image gene...,without rewrite+grade,1.0,1.000000,0
8,Show papers about reinforcement learning from ...,with rewrite+grade,0.0,0.000000,6
9,Show papers about reinforcement learning from ...,without rewrite+grade,1.0,1.000000,0


## Citation review sample

Call src.evaluate.citation_sample_csv for manual checking.

In [6]:
from src.evaluate import citation_sample_csv

citation_sample_csv(n=20, path=REPO_DIR / "eval" / "citation_sample.csv")

'/media/ibraheem/HDD/research_paper_discovery/eval/citation_sample.csv'

## Router accuracy (Sohaib)

Call src.evaluate.router_accuracy on the routing gold set.

In [7]:
import json, sys
sys.path.insert(0, str(REPO_DIR))
from src import nodes, evaluate
import pandas as pd

routing = json.load(open(REPO_DIR/"eval"/"eval_routing.json", encoding="utf-8"))
questions = [r["question"] for r in routing]
gold = [r["route"] for r in routing]

preds = []
for q in questions:
    st = nodes.router_node({"question": q, "llm_calls": 0})
    preds.append(st["route"])

ra = evaluate.router_accuracy(preds, gold)
print("router accuracy:", round(ra["accuracy"], 4))
print("n =", len(gold))

cm = ra["confusion"]
routes = list(cm.keys())
df_cm = pd.DataFrame(cm, index=routes, columns=routes).fillna(0).astype(int)
print(df_cm.to_string())

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(5,4))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Gold")
plt.title("Router confusion matrix")
plt.tight_layout()
plt.savefig(REPO_DIR/"images"/"10_router_confusion.png", dpi=120)
plt.close()
print("saved image")


router accuracy: 1.0
n = 40
            literature  trend  hybrid  fallback
literature          10      0       0         0
trend                0     10       0         0
hybrid               0      0      10         0
fallback             0      0       0        10


saved image


## Results

TODO: summarize the outputs and figures produced above.

## Stage 4 artifacts (Ibraheem)

These cells only **consume** values produced above. They do not edit `relevant_ids`, mark labels verified, or change any teammate cell. Every table derived from `eval_retrieval.json` is labelled `PRELIMINARY (unverified labels)` while any `needs_human_verification` is true.


In [8]:
# Stage 4 artifact generation (Ibraheem). Consumes the outputs above only.
import json
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

from src import rag
from src.evaluate import hit_rate_at_k, mrr

RESULTS = REPO_DIR / "results"
IMAGES = REPO_DIR / "images"
RESULTS.mkdir(parents=True, exist_ok=True)
IMAGES.mkdir(parents=True, exist_ok=True)

unverified = any(bool(item.get("needs_human_verification")) for item in retrieval_eval)
label_status = "PRELIMINARY (unverified labels)" if unverified else "verified"
print("label status:", label_status)
print("retrieval questions:", len(retrieval_eval),
      "unverified:", sum(bool(i.get("needs_human_verification")) for i in retrieval_eval))

# --- retrieval metrics: use the existing relevant_ids verbatim, never edit them ---
ranked = results
retrieval_gold = [list(item.get("relevant_ids", [])) for item in retrieval_eval]
hr5 = hit_rate_at_k(ranked, retrieval_gold, 5)
mrr_score = mrr(ranked, retrieval_gold)
pd.DataFrame([{
    "label_status": label_status,
    "n_questions": len(ranked),
    "hit_rate_at_5": round(hr5, 4),
    "mrr": round(mrr_score, 4),
    "needs_human_verification": unverified,
}]).to_csv(RESULTS / "phase2_retrieval_metrics.csv", index=False)
print("retrieval:", label_status, "hit_rate_at_5 =", round(hr5, 4), "mrr =", round(mrr_score, 4))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Hit Rate@5", "MRR"], [hr5, mrr_score], color=["#4c72b0", "#dd8452"])
ax.set_ylim(0, 1)
ax.set_ylabel("score")
ax.set_title(f"Retrieval — {label_status}")
for i, v in enumerate([hr5, mrr_score]):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center")
fig.tight_layout()
fig.savefig(IMAGES / "10_retrieval_verified.png", dpi=120)
plt.close(fig)

# --- ablation summary from the executed run_ablation table ---
abl = (ablation.groupby("variant", as_index=False)
       .agg(n_questions=("question", "size"),
            mean_hit_at_5=("hit@5", "mean"),
            mean_rr=("rr", "mean"),
            mean_llm_calls=("llm_calls", "mean")))
abl.insert(0, "label_status", label_status)
abl.to_csv(RESULTS / "phase2_ablation.csv", index=False)
ablation.to_csv(RESULTS / "phase2_ablation_questions.csv", index=False)
print(abl.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (col, title) in zip(axes, [("mean_hit_at_5", "Hit Rate@5"),
                                   ("mean_rr", "MRR"),
                                   ("mean_llm_calls", "avg llm_calls")]):
    ax.bar(abl["variant"], abl[col], color=["#4c72b0", "#dd8452"])
    ax.set_title(title)
    ax.tick_params(axis="x", labelrotation=15)
fig.suptitle(f"Ablation — {label_status}")
fig.tight_layout()
fig.savefig(IMAGES / "10_ablation.png", dpi=120)
plt.close(fig)

# --- router metrics from Sohaib's executed router section ---
router_metrics = pd.DataFrame([{
    "n_questions": len(questions),
    "accuracy": round(ra["accuracy"], 4),
    "correct": int(round(ra["accuracy"] * len(questions))),
}])
router_metrics.to_csv(RESULTS / "phase2_router_metrics.csv", index=False)
df_cm.to_csv(RESULTS / "phase2_router_confusion.csv")
print("router accuracy:", round(ra["accuracy"], 4), "n =", len(questions))

# --- candidate label review: ids/titles/abstracts, no label changes ---
papers = pd.read_parquet(DATA_PATH, columns=["id", "title", "abstract"])
by_id = papers.set_index("id")
review_note = (
    "Every `relevant_ids` entry in `eval/eval_retrieval.json` still has "
    "`needs_human_verification: true`. These are candidates only; no label was "
    "edited or marked verified by this notebook."
    if unverified
    else "Every `relevant_ids` entry in `eval/eval_retrieval.json` has "
         "`needs_human_verification: false`. Labels were reviewed in "
         "`eval/label_decisions.csv` and applied mechanically."
)
lines = [
    f"# Label review — {label_status}",
    "",
    review_note,
    "",
]
missing = 0
for item in retrieval_eval:
    lines.append(f"## {item['question']}")
    lines.append(f"filters: `{json.dumps(item.get('filters', {}), sort_keys=True)}`")
    lines.append("")
    lines.append("| arxiv_id | title | abstract (first 200 chars) |")
    lines.append("|---|---|---|")
    for aid in item.get("relevant_ids", []):
        if aid in by_id.index:
            row = by_id.loc[aid]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
            title = str(row["title"]).replace("|", "\\|")
            abstract = " ".join(str(row["abstract"]).split())[:200].replace("|", "\\|")
        else:
            missing += 1
            title, abstract = "NOT FOUND IN papers.parquet", ""
        lines.append(f"| {aid} | {title} | {abstract} |")
    lines.append("")
lines.append(f"Candidate ids not found in `papers.parquet`: {missing}.")
(RESULTS / "label_review.md").write_text("\n".join(lines), encoding="utf-8")
print("label review candidates:",
      sum(len(i.get("relevant_ids", [])) for i in retrieval_eval), "missing:", missing)

# --- exactly 20 citation-review rows, supported? left blank for a human ---
cite = pd.read_csv(REPO_DIR / "eval" / "citation_sample.csv")
review = cite.drop_duplicates(subset=["question", "arxiv_id"], keep="first").head(20).copy()
if len(review) < 20:
    extra = []
    for item in retrieval_eval:
        if len(review) + len(extra) >= 20:
            break
        q = item["question"]
        f = item.get("filters", {})
        docs = rag.retrieve(q, k=1, year_min=f.get("year_min"),
                            year_max=f.get("year_max"), category=f.get("category"),
                            topic=f.get("topic"))
        doc = docs[0] if docs else {}
        extra.append({"question": q, "answer": "",
                      "arxiv_id": doc.get("arxiv_id", ""),
                      "title": doc.get("title", ""), "supported?": ""})
    review = pd.concat([review, pd.DataFrame(extra)], ignore_index=True)
review = review.head(20).copy()
review["supported?"] = ""
review = review[["question", "answer", "arxiv_id", "title", "supported?"]]
review.to_csv(REPO_DIR / "eval" / "citation_sample.csv", index=False)
print("citation rows:", len(review),
      "blank supported?:", bool((review["supported?"].fillna("") == "").all()))

print("saved results/*.csv, results/label_review.md, images/10_*.png")


label status: verified
retrieval questions: 30 unverified: 0
retrieval: verified hit_rate_at_5 = 1.0 mrr = 0.9567


label_status               variant  n_questions  mean_hit_at_5  mean_rr  mean_llm_calls
    verified    with rewrite+grade           30            0.9 0.838889             4.2
    verified without rewrite+grade           30            1.0 0.898333             0.0


router accuracy: 1.0 n = 40


label review candidates: 465 missing: 0
citation rows: 20 blank supported?: True
saved results/*.csv, results/label_review.md, images/10_*.png


In [9]:
# Verified-label retrieval metrics at k=1/5/10 (Ibraheem).
from src.rag import retrieve

retrieved_at_10 = []
for item in retrieval_eval:
    filters = item.get("filters", {})
    docs = retrieve(
        item["question"], k=10,
        year_min=filters.get("year_min"), year_max=filters.get("year_max"),
        category=filters.get("category"), topic=filters.get("topic"),
    )
    retrieved_at_10.append([doc["arxiv_id"] for doc in docs])

gold_at_k = [list(item.get("relevant_ids", [])) for item in retrieval_eval]
metrics_at_k = {
    "label_status": label_status,
    "n_questions": len(retrieved_at_10),
    "hit_rate_at_1": round(hit_rate_at_k(retrieved_at_10, gold_at_k, 1), 4),
    "hit_rate_at_5": round(hit_rate_at_k(retrieved_at_10, gold_at_k, 5), 4),
    "hit_rate_at_10": round(hit_rate_at_k(retrieved_at_10, gold_at_k, 10), 4),
    "mrr": round(mrr([r[:5] for r in retrieved_at_10], gold_at_k), 4),
    "mrr_at_10": round(mrr(retrieved_at_10, gold_at_k), 4),
    "needs_human_verification": unverified,
}
pd.DataFrame([metrics_at_k]).to_csv(
    RESULTS / "phase2_retrieval_metrics_at_k.csv", index=False
)
print("retrieval @k:", metrics_at_k)


retrieval @k: {'label_status': 'verified', 'n_questions': 30, 'hit_rate_at_1': 0.9333, 'hit_rate_at_5': 1.0, 'hit_rate_at_10': 1.0, 'mrr': 0.9567, 'mrr_at_10': 0.9567, 'needs_human_verification': False}
